# Capstone --- Chapter 13: Governed Retrieval

The capstone answers a complaint by quoting policy, so the question is not what sounds relevant but what the policy actually says. This chapter adds the retrieval layer the agent's `search_policy` tool uses: a triple-mediated retriever over the calibrated policy store built in Chapter~2. It grounds every answer in a source, returns exact figures rather than paraphrases and refuses when nothing in the policy grounds the question. Retrieval here is a grounding discipline, not a similarity search.

## Loading the retriever

`get_default_retriever()` loads the shipped policy store and its calibrated gates. It is the same object `search_policy` calls at run time, so what this notebook shows is the agent's real retrieval path, not a stand-in. Loading it does not rebuild the store; the store is the artifact Chapter~2 produced.

In [1]:
import json
import forgeloop
from forgeloop.agents.capstone.policy_rag import get_default_retriever

retriever = get_default_retriever()
print('policy retriever loaded from', forgeloop.data_root().name, '/ gms_policy_store_cap')

knowlytix-core v0.2.0 licensed to customer=KnowlytixAgentBuilder tier=enterprise expires=2027-06-04


  GMS entities:  59
  GMS relations: 31
  GMS triples:   68
  Store loaded from beyond-prompt-and-pray/code/data/gms_policy_store_cap
  Entities:  59
  Relations: 31
  Triples:   68
  ENM:       27
  Documents: 0


<venv>/lib/python3.12/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  queued_call()


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

policy retriever loaded from data / gms_policy_store_cap


## Parse and bind: a complaint becomes a grounded query

Before anything is retrieved, the complaint is parsed into a query over policy entities and each term is bound to a real entity in the store. `extract` runs that parse-and-bind step and reports whether the query bound. A complaint about an unauthorized overdraft fee binds to the fee-reversal policy and the authorization it turns on, which is the query the retriever will answer.

In [2]:
complaint = ('I was charged a $35 overdraft fee I did not authorize, '
             'and I want it reversed.')
ex = retriever.extract(complaint)
print('bound to a real policy query:', ex['is_bound'])
for h, r, t in ex['query_facts']:
    print(f'  ({h}, {r}, {t})')

bound to a real policy query: True
  (fee_reversal, has_representative_reversal_cap_usd, ?)


## Routing: which policy domains a question grounds to

`route` returns the policy domains a query grounds to, ranked by plausibility and filtered by the store's gates, with no language-model synthesis. It is the honest answer to "where in policy does this live," and it is empty when the question grounds nowhere.

In [3]:
for q in ['what is the overdraft fee', 'who can authorize a fee reversal',
          'when does UDAAP apply', 'what is the weather today']:
    print(f'  {q:34s} -> {retriever.route(q)}')

The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  what is the overdraft fee          -> ['overdraft', 'fee_reversal']


  who can authorize a fee reversal   -> ['fee_reversal']


  when does UDAAP apply              -> ['regulatory_escalation', 'udaap']


  what is the weather today          -> []


## Grounded retrieval: the source, the exact figure, the provenance

`search` returns the grounded answer in the shape `search_policy` hands the agent: the policy domain it bound to and the passage that supports it, with the figure stated exactly as the policy states it. The dollar amounts here are read from the policy, not reconstructed by a model, so a reply built on them can be traced back to the sentence it came from.

In [4]:
questions = ['what is the overdraft fee',
             'who can authorize a fee reversal',
             'when does UDAAP apply',
             'account closure notice period']
for q in questions:
    hits = retriever.search(q, k=3)
    print(f'Q: {q}')
    for h in hits:
        passage = h['text'].split(chr(10))[0]
        print(f"   policy: {h['id']:24s} | {passage[:88]}")
    print()

Q: what is the overdraft fee
   policy: fee_reversal             | The Bank assesses an overdraft fee of $35.00 for each item paid into overdraft, charged 



Q: who can authorize a fee reversal
   policy: fee_reversal             | A customer-service representative may reverse a fee of up to $35.00 without further appr



Q: when does UDAAP apply
   policy: regulatory_escalation    | 1



Q: account closure notice period
   policy: account_closure          | When the Bank closes a consumer account on its own initiative, it provides the customer 



## Abstention: no grounding, no answer

A question the policy does not cover returns nothing. The retriever does not reach for the nearest chunk and let the model dress it up as an answer. An empty result is the correct outcome, and it is what lets the agent escalate or decline rather than invent a policy that does not exist.

In [5]:
off_topic = 'what is the weather today'
hits = retriever.search(off_topic, k=3)
print(f'Q: {off_topic}')
print('   routes :', retriever.route(off_topic))
print('   result :', hits, '(abstains -- no policy grounds this)')

Q: what is the weather today


   routes : []
   result : [] (abstains -- no policy grounds this)


## The layer the capstone assembles

This retriever is exactly what the `search_policy` tool wraps, so nothing here is repeated in Chapter~16: the assembled agent grounds a reply by calling `search`, quotes the exact figure it returns and escalates when it abstains. The store it reads was built and calibrated in Chapter~2; the gates that admit or reject a retrieved fact are the ones calibrated there.

In [6]:
# Self-check: a covered question grounds to an exact figure; an uncovered one abstains.
fee = retriever.search('what is the overdraft fee', k=3)
assert fee, 'expected a grounded fact for the overdraft fee'
assert '35' in fee[0]['text'], 'grounded passage should carry the exact $35 figure'
assert retriever.route('what is the overdraft fee'), 'covered query should route'
assert retriever.search('what is the weather today', k=3) == [], 'off-topic must abstain'
print('OK: governed retrieval grounds covered questions and abstains on the rest')

OK: governed retrieval grounds covered questions and abstains on the rest
